<a href="https://colab.research.google.com/github/ob3x/calorie-predictor/blob/main/calories_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import bibliotek oraz danych, wczytywanie danych oraz kopiowanie danych

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import kagglehub

# Download latest version
path = kagglehub.dataset_download("evgenyarbatov/polar-vantage-v-data")

np.random.seed(42)
np.printoptions(precision=3, suppress=True)

print("Path to dataset files:", path)

100%|██████████| 243k/243k [00:00<00:00, 21.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/evgenyarbatov/polar-vantage-v-data/versions/1


In [ ]:
raw_data = pd.read_csv(f"{path}/training_sessions.csv")
raw_data.columns

Index(['session_id', 'date', 'local_time', 'sport_id', 'sport_name',
       'duration_s', 'calories', 'hr_avg', 'hr_max', 'hr_min', 'training_load',
       'cardio_load', 'muscle_load', 'training_benefit', 'recovery_time_s',
       'carbo_pct', 'fat_pct', 'distance_m', 'ascent_m', 'descent_m',
       'speed_avg_ms', 'speed_max_ms', 'cadence_avg', 'cadence_max',
       'power_avg_w', 'power_max_w', 'running_index', 'weight_kg', 'vo2_max',
       'resting_hr', 'aerobic_threshold_hr', 'anaerobic_threshold_hr', 'age',
       'device'],
      dtype='object')

In [ ]:
df = raw_data.copy()
df.head()

,session_id,date,local_time,sport_id,sport_name,duration_s,calories,hr_avg,hr_max,hr_min,...,power_avg_w,power_max_w,running_index,weight_kg,vo2_max,resting_hr,aerobic_threshold_hr,anaerobic_threshold_hr,age,device
0,1,2019-11-19,21:15:51,15,Road Running,613.734,46.0,97.0,122.0,70.0,...,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,30,NaN
1,2,2019-11-20,06:50:24,15,Road Running,1000.398,107.0,108.0,138.0,71.0,...,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,30,NaN
2,3,2019-11-20,21:57:20,15,Road Running,1537.893,104.0,95.0,132.0,69.0,...,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,30,NaN
3,4,2019-11-21,22:48:51,15,Road Running,1187.993,100.0,100.0,129.0,66.0,...,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,30,NaN
4,5,2019-11-22,21:03:20,15,Road Running,641.184,50.0,98.0,126.0,71.0,...,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,30,NaN


##Usuwanie kolumn niepotrzebnych, oraz tych z dużą ilością braków danych, zmienianie typu danych na kategoryczny

In [ ]:
df.drop(labels=["session_id","hr_max", "hr_min", "device", "descent_m","distance_m", "speed_avg_ms", "cadence_avg", "ascent_m", "cadence_max","speed_max_ms", "running_index", "muscle_load", "power_avg_w", "power_max_w", "date", "local_time", "sport_id", "training_benefit"], axis=1, inplace=True)
df["sport_name"] = df["sport_name"].astype("category")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2258 entries, 0 to 2257
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   sport_name              2258 non-null   category
 1   duration_s              2258 non-null   float64 
 2   calories                2252 non-null   float64 
 3   hr_avg                  2251 non-null   float64 
 4   training_load           2185 non-null   float64 
 5   cardio_load             2232 non-null   float64 
 6   recovery_time_s         2251 non-null   float64 
 7   carbo_pct               2185 non-null   float64 
 8   fat_pct                 2244 non-null   float64 
 9   weight_kg               2258 non-null   float64 
 10  vo2_max                 2210 non-null   float64 
 11  resting_hr              2189 non-null   float64 
 12  aerobic_threshold_hr    2186 non-null   float64 
 13  anaerobic_threshold_hr  2186 non-null   float64 
 14  age                     

#Podejrzenie braków danych, usuwanie wierszy z brakami w zmiennej docelowej, uzupełnianie braków danych średnią

In [ ]:
df = df.dropna(subset=["calories"])
df.isnull().sum()

,0
sport_name,0
duration_s,0
calories,0
hr_avg,2
training_load,67
cardio_load,20
recovery_time_s,1
carbo_pct,67
fat_pct,8
weight_kg,0


In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)
df.isnull().sum()

,0
sport_name,0
duration_s,0
calories,0
hr_avg,0
training_load,0
cardio_load,0
recovery_time_s,0
carbo_pct,0
fat_pct,0
weight_kg,0


In [ ]:
df.head()

,sport_name,duration_s,calories,hr_avg,training_load,cardio_load,recovery_time_s,carbo_pct,fat_pct,weight_kg,vo2_max,resting_hr,aerobic_threshold_hr,anaerobic_threshold_hr,age
0,Road Running,613.734,46.0,97.0,98.680092,3.92458,780.0,56.394966,50.0,66.0,59.792478,55.0,141.490393,169.428637,30
1,Road Running,1000.398,107.0,108.0,98.680092,9.48221,3120.0,56.394966,40.0,66.0,59.792478,55.0,141.490393,169.428637,30
2,Road Running,1537.893,104.0,95.0,98.680092,9.27808,1320.0,56.394966,51.0,66.0,59.792478,55.0,141.490393,169.428637,30
3,Road Running,1187.993,100.0,100.0,98.680092,8.61795,1680.0,56.394966,47.0,66.0,59.792478,55.0,141.490393,169.428637,30
4,Road Running,641.184,50.0,98.0,98.680092,4.35936,840.0,56.394966,48.0,66.0,59.792478,55.0,141.490393,169.428637,30


##Oddzielenie zmiennej docelowej od ramki danych

In [ ]:
target = df["calories"]
data = df.drop("calories", axis=1)

print(f"Target shape: {target.shape}")
print(f"Data shape: {data.shape}")

Target shape: (2252,)
Data shape: (2252, 14)


##Podział danych na testowe i treningowe

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1801, 14)
y_train shape: (1801,)
X_test shape: (451, 14)
y_test shape: (451,)


##Kodowanie zmiennych kategorycznych

In [ ]:
X_train = pd.get_dummies(X_train, columns=["sport_name"], dtype=int, drop_first=True)
X_test = pd.get_dummies(X_test, columns=["sport_name"], dtype=int, drop_first=True)

x_columns = X_train.columns
x_columns

Index(['duration_s', 'hr_avg', 'training_load', 'cardio_load',
       'recovery_time_s', 'carbo_pct', 'fat_pct', 'weight_kg', 'vo2_max',
       'resting_hr', 'aerobic_threshold_hr', 'anaerobic_threshold_hr', 'age',
       'sport_name_Cycling', 'sport_name_Gym / Fitness', 'sport_name_Hiking',
       'sport_name_Indoor Cycling', 'sport_name_Indoor Training',
       'sport_name_Other', 'sport_name_Road Running', 'sport_name_Rowing',
       'sport_name_Running', 'sport_name_Strength Training',
       'sport_name_Swimming', 'sport_name_Triathlon', 'sport_name_Walking',
       'sport_name_Yoga / Flexibility'],
      dtype='object')

##Standaryzacja

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

##Model liniowy

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

LinearRegression()

In [ ]:
y_pred = lin_reg.predict(X_test)
lin_reg_y_error = pd.DataFrame({"y_true" : y_test, "y_pred" : y_pred})
lin_reg_y_error["error"] = lin_reg_y_error["y_true"] - lin_reg_y_error["y_pred"]
lin_reg_y_error["squared_error"] = lin_reg_y_error["error"]**2
lin_reg_y_error.head()

,y_true,y_pred,error,squared_error
1602,838.0,882.796164,-44.796164,2006.696281
646,713.0,719.960403,-6.960403,48.447205
810,304.0,321.041652,-17.041652,290.417891
1177,1176.0,1285.237297,-109.237297,11932.787086
940,782.0,774.050571,7.949429,63.193420


###Ocena modelu

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2_score: {r2_score(y_test, y_pred)}")

MSE: 21740.4953
RMSE: 147.4466
MAE: 32.9860
R2_score: 0.9566430900767057


##Regresja drzew decyzyjnych

In [ ]:
from sklearn.tree import DecisionTreeRegressor

regressor = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5)
regressor.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=7, min_samples_leaf=5)

###Ocena modelu na zbiorze testowym oraz treningowym

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_test = regressor.predict(X_test)
y_pred_train = regressor.predict(X_train)


print(f"MSE-train: {mean_squared_error(y_train, y_pred_train):.4f}")
print(f"RMSE-train: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
print(f"MAE-train: {mean_absolute_error(y_train, y_pred_train):.4f}")
print(f"R2_score-train: {r2_score(y_train, y_pred_train)}")
print("--------------------")
print(f"MSE-test: {mean_squared_error(y_test, y_pred_test):.4f}")
print(f"RMSE-test: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"MAE-test: {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"R2_score-test: {r2_score(y_test, y_pred_test)}")

MSE-train: 17011.2239
RMSE-train: 130.4271
MAE-train: 32.1438
R2_score-train: 0.9671801558174961
--------------------
MSE-test: 11569.4260
RMSE-test: 107.5613
MAE-test: 42.0742
R2_score-test: 0.9769271787165884


##Sprawdzanie niepotrzebnych cech

In [ ]:
importances = regressor.feature_importances_

for feature, score in zip(x_columns, importances):
    print(f"{feature}: {score:.4f}")

duration_s: 0.0385
hr_avg: 0.0001
training_load: 0.0515
cardio_load: 0.8595
recovery_time_s: 0.0493
carbo_pct: 0.0000
fat_pct: 0.0000
weight_kg: 0.0000
vo2_max: 0.0000
resting_hr: 0.0000
aerobic_threshold_hr: 0.0000
anaerobic_threshold_hr: 0.0000
age: 0.0009
sport_name_Cycling: 0.0000
sport_name_Gym / Fitness: 0.0000
sport_name_Hiking: 0.0000
sport_name_Indoor Cycling: 0.0000
sport_name_Indoor Training: 0.0000
sport_name_Other: 0.0000
sport_name_Road Running: 0.0000
sport_name_Rowing: 0.0000
sport_name_Running: 0.0000
sport_name_Strength Training: 0.0000
sport_name_Swimming: 0.0000
sport_name_Triathlon: 0.0000
sport_name_Walking: 0.0000
sport_name_Yoga / Flexibility: 0.0000


###Usuwanie niepotrzebnych cech

In [ ]:
new_data = data.drop(labels=["sport_name", "anaerobic_threshold_hr", "aerobic_threshold_hr", "resting_hr", "vo2_max", "weight_kg", "fat_pct", "carbo_pct"], axis=1)
new_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2252 entries, 0 to 2257
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   duration_s       2252 non-null   float64
 1   hr_avg           2252 non-null   float64
 2   training_load    2252 non-null   float64
 3   cardio_load      2252 non-null   float64
 4   recovery_time_s  2252 non-null   float64
 5   age              2252 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 123.2 KB


###Podział na zbiór testowy oraz treningowy dla przefiltrowanych danych


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(new_data, target, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (1801, 6)
y_train shape: (1801,)
X_test shape: (451, 6)
y_test shape: (451,)


####Regresja drzew decyzyjnych oraz ocena modelu

In [ ]:
from sklearn.tree import DecisionTreeRegressor

new_regressor = DecisionTreeRegressor(max_depth=8, min_samples_leaf=5)
new_regressor.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=8, min_samples_leaf=5)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred_test = new_regressor.predict(X_test)
y_pred_train = new_regressor.predict(X_train)


print(f"MSE-train: {mean_squared_error(y_train, y_pred_train):.4f}")
print(f"RMSE-train: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
print(f"MAE-train: {mean_absolute_error(y_train, y_pred_train):.4f}")
print(f"R2_score-train: {r2_score(y_train, y_pred_train)}")
print("--------------------")
print(f"MSE-test: {mean_squared_error(y_test, y_pred_test):.4f}")
print(f"RMSE-test: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"MAE-test: {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"R2_score-test: {r2_score(y_test, y_pred_test)}")

MSE-train: 16665.5417
RMSE-train: 129.0951
MAE-train: 27.6651
R2_score-train: 0.9678470824297805
--------------------
MSE-test: 10690.1175
RMSE-test: 103.3930
MAE-test: 36.5527
R2_score-test: 0.9786807772890784


In [ ]:
import joblib

model_columns = list(x_columns)
joblib.dump(new_regressor, 'model_drzewa.pkl')

['model_drzewa.pkl']